<div class="alert alert-block alert-success">


# **03 — feature engineering pipeline**

## **Goal of this notebook**
The raw columns in the database (goals, minutes_played, date_of_birth, net_transfer_record) are not directly comparable or analytically useful on their own. This notebook derives meaningful metrics from the raw data so that every subsequent notebook has better inputs to work with. This is not analysis — it is preparation for analysis.

### **Some metrics to make**
Player-level metrics
- age — calculated from date_of_birth to a reference date (e.g. latest game date in dataset)
- contract_years_remaining — from contract_expiration_date to reference date
- goals_per_90 — goals / (minutes_played / 90), for players with sufficient appearances
- assists_per_90 — same approach
- goal_contributions_per_90 — (goals + assists) / (minutes_played / 90)
- minutes_played_ratio — minutes_played / (appearances × 90), a proxy for how often a player plays the full match vs gets substituted
- cards_per_90 — (yellow_cards + red_cards×2) / (minutes_played / 90)

Club-level metrics (aggregated from games and appearances)
- win_rate — wins / total_games
- goals_scored_per_game — total goals scored / total games
- goals_conceded_per_game — total goals conceded / total games
- goal_difference_per_game
- avg_attendance — average attendance across home games

Game-level metrics (only row-level metrics not aggregated)
- total_goals — home_club_goals + away_club_goals
- goal_difference — absolute difference between home and away goals
- is_draw, home_win, away_win — boolean outcome flags

##### **the changes are going to take place directly on the database not the raw data**

In [1]:
# importing the necessaries
import sys
import os
# Adding the root to the path to use utils folder
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.db_utils import run_query , execute_ddl
from utils.custom_plots import distribution_plot, outlier_plot
from utils.schema_diagram import schema_diagram


'''import plotly.io as pio
pio.renderers.default = "png" ''' # drop these 2 lines if you want interactive charts locally

'import plotly.io as pio\npio.renderers.default = "png" '

<div class="alert alert-block alert-success">

# First lets see the connections 
## here we can have a better understanding of cross-table metrics

In [9]:
database_diagram=schema_diagram(run_query_fn=run_query,
                                layout='circular',
                                edge_labels='always',
                                title='Football database schema',
                                edge_label_font_size=13
                                )

database_diagram

<div class="alert alert-block alert-success">

# Player-level metrics:


In [ ]:
# creating age column for players:


# we calculate the age based on the latest data which is 2025(not 2026) 
age_query = r'''
ALTER TABLE players
ADD COLUMN age INT GENERATED ALWAYS AS (
extract(year from age(cast('2025-01-01' as date), date_of_birth))
) STORED; 
'''

contract_remaining_date_query = r'''
ALTER TABLE players
ADD COLUMN contract_years_remaining NUMERIC(4, 1) GENERATED ALWAYS AS (
    ROUND(
        EXTRACT(EPOCH FROM AGE(contract_expiration_date, CAST('2025-01-01' AS DATE)))
        / (365.25 * 24 * 3600)
    , 1)
) STORED;
'''


# we write a query for metrics that need aggregation  
# GENERATED ALWAYS AS doesnt work for metrics that need computation on more then 1 row so instead we define columns and populate them with computed value
# we need to define a threshold for ratio denominator so it doesnt explode the ratio with low values 

player_aggregated_metrics_query = r'''
ALTER TABLE players
ADD COLUMN goals_per_90            NUMERIC(5, 2),
ADD COLUMN assists_per_90          NUMERIC(5, 2),
ADD COLUMN goal_contributions_per_90 NUMERIC(5, 2),
ADD COLUMN minutes_played_ratio    NUMERIC(4, 3),
ADD COLUMN cards_per_90            NUMERIC(5, 2),
ADD COLUMN total_appearances       INT,
ADD COLUMN total_minutes_played    INT;


UPDATE players p
SET
    total_appearances        = agg.appearances,
    total_minutes_played     = agg.total_minutes,
    goals_per_90             = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND(agg.total_goals / (agg.total_minutes / 90.0), 2) 
                               END,
    assists_per_90           = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND(agg.total_assists / (agg.total_minutes / 90.0), 2) 
                               END,
    goal_contributions_per_90 = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND((agg.total_goals + agg.total_assists) / (agg.total_minutes / 90.0), 2) 
                               END,
    minutes_played_ratio     = CASE 
                                 WHEN agg.appearances > 5
                                 THEN ROUND(agg.total_minutes / (agg.appearances * 90.0), 3) 
                               END,
    cards_per_90             = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND((agg.yellow_cards + agg.red_cards * 2) / (agg.total_minutes / 90.0), 2) 
                               END
FROM (
    SELECT
        player_id,
        COUNT(*)             AS appearances,
        SUM(minutes_played)  AS total_minutes,
        SUM(goals)           AS total_goals,
        SUM(assists)         AS total_assists,
        SUM(yellow_cards)    AS yellow_cards,
        SUM(red_cards)       AS red_cards
    FROM appearances
    GROUP BY player_id
) agg
WHERE p.player_id = agg.player_id;
'''


<div class="alert alert-block alert-success">

## Player-level metrics summary:

- age — the player's current age in completed years as of the reference date
- contract_years_remaining — how many years (as a decimal) until the player's contract expires, negative means already expired
- goals_per_90 — how many goals the player scores on average per 90 minutes of play
- assists_per_90 — how many assists the player produces on average per 90 minutes of play
- goal_contributions_per_90 — combined goals and assists per 90 minutes, measuring total attacking output regardless of who finished
- minutes_played_ratio — what fraction of available minutes the player actually plays, where 1.0 means they play every minute of every game they appear in, values above 1.0 are unlikely but possible it means the players average play time is more than 90 minute because of extra times 
- cards_per_90 — disciplinary load per 90 minutes, treating a red card as twice the weight of a yellow
- total_appearances — total number of games the player appeared in across all recorded seasons
- total_minutes_played — total minutes across all appearances, the denominator behind every per-90 metric

<div class="alert alert-block alert-success">

# Game-level metrics:


In [ ]:
game_metrics_query = r'''
ALTER TABLE games
ADD COLUMN total_goals INT GENERATED ALWAYS AS (
    home_club_goals + away_club_goals
) STORED;

ALTER TABLE games
ADD COLUMN goal_difference INT GENERATED ALWAYS AS (
    ABS(home_club_goals - away_club_goals)
) STORED;

ALTER TABLE games
ADD COLUMN is_draw BOOLEAN GENERATED ALWAYS AS (
    home_club_goals = away_club_goals
) STORED;

ALTER TABLE games
ADD COLUMN home_win BOOLEAN GENERATED ALWAYS AS (
    home_club_goals > away_club_goals
) STORED;

ALTER TABLE games
ADD COLUMN away_win BOOLEAN GENERATED ALWAYS AS (
    home_club_goals < away_club_goals
) STORED;
'''
# we dont need aggregated metrics for game-level metrics, these metrics make querying easier and cleaner

<div class="alert alert-block alert-success">

## Game-level metrics summary:

- total_goals — combined goals from both teams in a single match, a measure of how open or entertaining the game was
- goal_difference — absolute gap between the two teams' scores, measuring how decisive the result was
- is_draw — whether the match ended level
- home_win — whether the home team won
- away_win — whether the away team won

<div class="alert alert-block alert-success">

# club-level metrics:


In [ ]:
# first lets define the row-level metric 

club_metrics_query = r'''
ALTER TABLE clubs
ADD COLUMN foreigners_ratio  NUMERIC(4, 3) GENERATED ALWAYS AS (
    ROUND(foreigners_number::NUMERIC / NULLIF(squad_size, 0), 3)
) STORED;
'''

aggregated_club_metrics_query = r'''
ALTER TABLE clubs
ADD COLUMN win_rate               NUMERIC(4, 3),
ADD COLUMN goals_scored_per_game  NUMERIC(5, 2),
ADD COLUMN goals_conceded_per_game NUMERIC(5, 2),
ADD COLUMN goal_difference_per_game NUMERIC(5, 2),
ADD COLUMN avg_attendance         NUMERIC(10, 1),
ADD COLUMN total_games            INT;

UPDATE clubs c
SET
    total_games              = agg.total_games,
    win_rate                 = ROUND(agg.wins::NUMERIC / NULLIF(agg.total_games, 0), 3),
    goals_scored_per_game    = ROUND(agg.goals_scored::NUMERIC / NULLIF(agg.total_games, 0), 2),
    goals_conceded_per_game  = ROUND(agg.goals_conceded::NUMERIC / NULLIF(agg.total_games, 0), 2),
    goal_difference_per_game = ROUND((agg.goals_scored - agg.goals_conceded)::NUMERIC / NULLIF(agg.total_games, 0), 2),
    avg_attendance           = ROUND(agg.avg_attendance, 1)
FROM (
    SELECT
        club_id,
        COUNT(*)                  AS total_games,
        SUM(goals_scored)         AS goals_scored,
        SUM(goals_conceded)       AS goals_conceded,
        SUM(wins)                 AS wins,
        AVG(attendance)           AS avg_attendance
    FROM (
        -- club as home team
        SELECT
            home_club_id        AS club_id,
            home_club_goals     AS goals_scored,
            away_club_goals     AS goals_conceded,
            CASE WHEN home_club_goals > away_club_goals THEN 1 ELSE 0 END AS wins,
            attendance
        FROM games
        WHERE home_club_goals IS NOT NULL

        UNION ALL      -- to catch both home-away teams stats

        -- club as away team
        SELECT
            away_club_id        AS club_id,
            away_club_goals     AS goals_scored,
            home_club_goals     AS goals_conceded,
            CASE WHEN away_club_goals > home_club_goals THEN 1 ELSE 0 END AS wins,
            NULL                AS attendance  -- attendance belongs to home club only
        FROM games
        WHERE away_club_goals IS NOT NULL
    ) both_sides
    GROUP BY club_id
) agg
WHERE c.club_id = agg.club_id;
'''

In [2]:
clubs_df = run_query('select * from clubs')
clubs_df

,club_id,name,domestic_competition_id,squad_size,foreigners_number,national_team_players,stadium_name,stadium_seats,net_transfer_record,numeric_net_transfer_record
0,2578,Saint Johnstone Football Club,SC1,25,16,2,McDiarmid Park,10696,+-0,0.0
1,2727,Oud-Heverlee Leuven,BE1,29,16,2,King Power at Den Dreef Stadion,10020,+300k,300000.0
2,28643,SK Beveren,BE1,27,16,3,Freethielstadion,13290,-350k,-350000.0
3,2990,Académica Coimbra,PO1,26,6,0,Estádio Cidade de Coimbra,29744,+150k,150000.0
4,3057,Royal Standard Club de Liège,BE1,27,18,10,Maurice Dufrasne Stadion,27670,+3.11m,3110000.0
...,...,...,...,...,...,...,...,...,...,...
421,68608,CF Os Belenenses,PO1,30,5,0,Estádio do Restelo,19980,+-0,0.0
422,724,Football Club Volendam,NL1,32,17,3,Kras Stadion,7384,+2.30m,2300000.0
423,800,Atalanta Bergamasca Calcio S.p.a.,IT1,25,17,16,Gewiss Stadium,21747,+74.49m,74490000.0
424,979,Moreirense Futebol Clube,PO1,29,17,3,Estádio C. J. de Almeida Freitas,6153,-2.05m,-2050000.0


In [3]:
ap_df = run_query('select * from appearances')
ap_df

,appearance_id,game_id,player_id,yellow_cards,red_cards,goals,assists,minutes_played
0,3393960_546880,3393960,546880,0,0,0,0,48
1,3393960_57370,3393960,57370,0,0,1,0,90
2,3393960_640428,3393960,640428,0,0,0,1,26
3,3393960_73734,3393960,73734,0,0,0,0,90
4,3393960_84301,3393960,84301,0,0,0,0,90
...,...,...,...,...,...,...,...,...
523233,3393960_435485,3393960,435485,0,0,0,0,8
523234,3393960_442531,3393960,442531,0,0,0,0,90
523235,3393960_442891,3393960,442891,0,0,0,0,90
523236,3393960_475413,3393960,475413,0,1,0,0,35
